In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (accuracy_score, precision_score,
                              recall_score, f1_score,
                              confusion_matrix, classification_report)
import warnings
warnings.filterwarnings('ignore')

# Load data
df = pd.read_csv("../data/churn_scaled.csv")

# Split X and y
X = df.drop('churn', axis=1)
y = df['churn']

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("✅ Setup complete!")
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

✅ Setup complete!
Train: (5625, 30), Test: (1407, 30)


In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score,
                              recall_score, f1_score,
                              confusion_matrix, classification_report)
import warnings
warnings.filterwarnings('ignore')

print("✅ Libraries imported!")

✅ Libraries imported!


In [2]:
# Load ML-ready dataset
df = pd.read_csv("../data/churn_scaled.csv")

print(f"Dataset shape: {df.shape}")
print(f"Churn distribution:\n{df['churn'].value_counts()}")

# Separate features and target
X = df.drop('churn', axis=1)
y = df['churn']

print(f"\nX shape: {X.shape}")
print(f"y shape: {y.shape}")

Dataset shape: (7032, 31)
Churn distribution:
churn
0    5163
1    1869
Name: count, dtype: int64

X shape: (7032, 30)
y shape: (7032,)


In [3]:
# 80/20 split with stratify (guarantees both classes in train AND test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y        # ← Ensures balanced split
)

print("✅ Train/Test split complete!")
print(f"\nTraining set:  {X_train.shape[0]} customers")
print(f"Testing set:   {X_test.shape[0]} customers")

print(f"\nTraining churn distribution:")
print(y_train.value_counts())

print(f"\nTesting churn distribution:")
print(y_test.value_counts())

✅ Train/Test split complete!

Training set:  5625 customers
Testing set:   1407 customers

Training churn distribution:
churn
0    4130
1    1495
Name: count, dtype: int64

Testing churn distribution:
churn
0    1033
1     374
Name: count, dtype: int64


In [4]:
# Initialize and train
lr_model = LogisticRegression(random_state=42, max_iter=1000)
lr_model.fit(X_train, y_train)

print("✅ Logistic Regression trained!")

# Show what the model learned (feature weights)
weights = pd.DataFrame({
    'Feature': X.columns,
    'Weight': lr_model.coef_[0].round(3)
}).sort_values('Weight', ascending=False)

print("\n🧠 Feature Weights (what the model learned):")
print(weights.to_string(index=False))

✅ Logistic Regression trained!

🧠 Feature Weights (what the model learned):
                              Feature  Weight
                         totalcharges   1.234
          internetservice_Fiber optic   0.835
       paymentmethod_Electronic check   0.391
                    multiplelines_Yes   0.306
                 paperlessbilling_Yes   0.289
                      streamingtv_Yes   0.266
                  streamingmovies_Yes   0.238
       multiplelines_No phone service   0.208
                        seniorcitizen   0.189
           paymentmethod_Mailed check   0.104
paymentmethod_Credit card (automatic)   0.038
                 deviceprotection_Yes   0.022
                          gender_Male  -0.025
                          partner_Yes  -0.034
                       monthlycharges  -0.072
  streamingmovies_No internet service  -0.119
     onlinebackup_No internet service  -0.119
      streamingtv_No internet service  -0.119
                   internetservice_No  -0.119
   o

In [5]:
# Predict on test data
y_pred = lr_model.predict(X_test)
y_prob = lr_model.predict_proba(X_test)

print("🎯 Sample Predictions (first 10):\n")

sample = pd.DataFrame({
    'Actual':           y_test.values[:10],
    'Predicted':        y_pred[:10],
    'Prob (Stay)':      y_prob[:10, 0].round(3),
    'Prob (Churn)':     y_prob[:10, 1].round(3),
    'Correct?':         ['✅' if a == p else '❌' 
                         for a, p in zip(y_test.values[:10], y_pred[:10])]
})

print(sample.to_string(index=False))

🎯 Sample Predictions (first 10):

 Actual  Predicted  Prob (Stay)  Prob (Churn) Correct?
      0          0        0.980         0.020        ✅
      0          1        0.411         0.589        ❌
      0          0        0.992         0.008        ✅
      1          0        0.807         0.193        ❌
      0          0        0.907         0.093        ✅
      1          0        0.543         0.457        ❌
      0          0        0.969         0.031        ✅
      0          0        0.836         0.164        ✅
      1          1        0.327         0.673        ✅
      0          0        0.980         0.020        ✅


In [6]:
print("📊 MODEL PERFORMANCE — LOGISTIC REGRESSION:\n")

accuracy  = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, zero_division=0)
recall    = recall_score(y_test, y_pred, zero_division=0)
f1        = f1_score(y_test, y_pred, zero_division=0)

print(f"Accuracy:  {accuracy:.2%}")
print(f"Precision: {precision:.2%}")
print(f"Recall:    {recall:.2%}")
print(f"F1-Score:  {f1:.2%}")

# Confusion matrix
print(f"\n🔢 CONFUSION MATRIX:\n")
cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
cm_df = pd.DataFrame(
    cm,
    index=['Actual: Stay (0)', 'Actual: Churn (1)'],
    columns=['Predicted: Stay (0)', 'Predicted: Churn (1)']
)
print(cm_df)

# Full report
print(f"\n📋 FULL CLASSIFICATION REPORT:\n")
print(classification_report(y_test, y_pred,
                             labels=[0, 1],
                             target_names=['Stay (0)', 'Churn (1)'],
                             zero_division=0))

📊 MODEL PERFORMANCE — LOGISTIC REGRESSION:

Accuracy:  80.31%
Precision: 64.74%
Recall:    56.95%
F1-Score:  60.60%

🔢 CONFUSION MATRIX:

                   Predicted: Stay (0)  Predicted: Churn (1)
Actual: Stay (0)                   917                   116
Actual: Churn (1)                  161                   213

📋 FULL CLASSIFICATION REPORT:

              precision    recall  f1-score   support

    Stay (0)       0.85      0.89      0.87      1033
   Churn (1)       0.65      0.57      0.61       374

    accuracy                           0.80      1407
   macro avg       0.75      0.73      0.74      1407
weighted avg       0.80      0.80      0.80      1407



161 missed churners × $756 lost revenue = $121,716 in preventable losses
116 false alarms   × $50 wasted effort  = $5,800 wasted

Total cost of model errors = $127,516

Logistic Regression Baseline Results:
- Accuracy 80% is misleading due to class imbalance (73/27 split)
- Recall of 57% is unacceptable for churn — missing 43% of churners
- Model is biased toward predicting "Stay" (the majority class)
- Next steps: Try Random Forest and XGBoost, consider class_weight='balanced'
- Target: Recall > 75% before deploying any model

In [1]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

print("✅ Decision Tree and Random Forest imported!")

✅ Decision Tree and Random Forest imported!


In [4]:
# Initialize Decision Tree
dt_model = DecisionTreeClassifier(
    random_state=42,
    max_depth=5        # Limits tree depth to prevent overfitting
)

# Train
dt_model.fit(X_train, y_train)

# Predict
dt_pred = dt_model.predict(X_test)

print("✅ Decision Tree trained!")
print(f"\nTree depth: {dt_model.get_depth()}")
print(f"Number of leaves: {dt_model.get_n_leaves()}")

✅ Decision Tree trained!

Tree depth: 5
Number of leaves: 32


In [5]:
print("📊 MODEL PERFORMANCE — DECISION TREE:\n")

dt_accuracy  = accuracy_score(y_test, dt_pred)
dt_precision = precision_score(y_test, dt_pred, zero_division=0)
dt_recall    = recall_score(y_test, dt_pred, zero_division=0)
dt_f1        = f1_score(y_test, dt_pred, zero_division=0)

print(f"Accuracy:  {dt_accuracy:.2%}")
print(f"Precision: {dt_precision:.2%}")
print(f"Recall:    {dt_recall:.2%}")
print(f"F1-Score:  {dt_f1:.2%}")

print(f"\n🔢 CONFUSION MATRIX:\n")
dt_cm = confusion_matrix(y_test, dt_pred, labels=[0, 1])
dt_cm_df = pd.DataFrame(
    dt_cm,
    index=['Actual: Stay (0)', 'Actual: Churn (1)'],
    columns=['Predicted: Stay (0)', 'Predicted: Churn (1)']
)
print(dt_cm_df)

print(f"\n📋 FULL CLASSIFICATION REPORT:\n")
print(classification_report(y_test, dt_pred,
                             labels=[0, 1],
                             target_names=['Stay (0)', 'Churn (1)'],
                             zero_division=0))

📊 MODEL PERFORMANCE — DECISION TREE:

Accuracy:  77.83%
Precision: 58.07%
Recall:    59.63%
F1-Score:  58.84%

🔢 CONFUSION MATRIX:

                   Predicted: Stay (0)  Predicted: Churn (1)
Actual: Stay (0)                   872                   161
Actual: Churn (1)                  151                   223

📋 FULL CLASSIFICATION REPORT:

              precision    recall  f1-score   support

    Stay (0)       0.85      0.84      0.85      1033
   Churn (1)       0.58      0.60      0.59       374

    accuracy                           0.78      1407
   macro avg       0.72      0.72      0.72      1407
weighted avg       0.78      0.78      0.78      1407



In [6]:
# Initialize Random Forest
rf_model = RandomForestClassifier(
    n_estimators=100,   # Number of trees
    random_state=42,
    max_depth=10        # Depth limit per tree
)

# Train
rf_model.fit(X_train, y_train)

# Predict
rf_pred = rf_model.predict(X_test)

print("✅ Random Forest trained!")
print(f"Number of trees: {rf_model.n_estimators}")

✅ Random Forest trained!
Number of trees: 100


In [7]:
print("📊 MODEL PERFORMANCE — RANDOM FOREST:\n")

rf_accuracy  = accuracy_score(y_test, rf_pred)
rf_precision = precision_score(y_test, rf_pred, zero_division=0)
rf_recall    = recall_score(y_test, rf_pred, zero_division=0)
rf_f1        = f1_score(y_test, rf_pred, zero_division=0)

print(f"Accuracy:  {rf_accuracy:.2%}")
print(f"Precision: {rf_precision:.2%}")
print(f"Recall:    {rf_recall:.2%}")
print(f"F1-Score:  {rf_f1:.2%}")

print(f"\n🔢 CONFUSION MATRIX:\n")
rf_cm = confusion_matrix(y_test, rf_pred, labels=[0, 1])
rf_cm_df = pd.DataFrame(
    rf_cm,
    index=['Actual: Stay (0)', 'Actual: Churn (1)'],
    columns=['Predicted: Stay (0)', 'Predicted: Churn (1)']
)
print(rf_cm_df)

print(f"\n📋 FULL CLASSIFICATION REPORT:\n")
print(classification_report(y_test, rf_pred,
                             labels=[0, 1],
                             target_names=['Stay (0)', 'Churn (1)'],
                             zero_division=0))

📊 MODEL PERFORMANCE — RANDOM FOREST:

Accuracy:  78.89%
Precision: 62.88%
Recall:    50.27%
F1-Score:  55.87%

🔢 CONFUSION MATRIX:

                   Predicted: Stay (0)  Predicted: Churn (1)
Actual: Stay (0)                   922                   111
Actual: Churn (1)                  186                   188

📋 FULL CLASSIFICATION REPORT:

              precision    recall  f1-score   support

    Stay (0)       0.83      0.89      0.86      1033
   Churn (1)       0.63      0.50      0.56       374

    accuracy                           0.79      1407
   macro avg       0.73      0.70      0.71      1407
weighted avg       0.78      0.79      0.78      1407



In [8]:
print("🌟 FEATURE IMPORTANCE — What drives churn?\n")

importance_df = pd.DataFrame({
    'Feature': X.columns,
    'Importance': rf_model.feature_importances_
}).sort_values('Importance', ascending=False)

print(importance_df.to_string(index=False))

print(f"\nTop 5 most important features:")
print(importance_df.head(5).to_string(index=False))

🌟 FEATURE IMPORTANCE — What drives churn?

                              Feature  Importance
                               tenure    0.182865
                         totalcharges    0.160851
                       monthlycharges    0.124068
          internetservice_Fiber optic    0.064206
                    contract_Two year    0.059125
       paymentmethod_Electronic check    0.057469
                   onlinesecurity_Yes    0.037121
                    contract_One year    0.033782
                      techsupport_Yes    0.027195
                 paperlessbilling_Yes    0.021329
 deviceprotection_No internet service    0.017181
                     onlinebackup_Yes    0.016932
                          partner_Yes    0.016350
                    multiplelines_Yes    0.015693
                       dependents_Yes    0.015391
                          gender_Male    0.015301
                        seniorcitizen    0.014782
   onlinesecurity_No internet service    0.013494
       

In [9]:
print("📊 MODEL COMPARISON — ALL MODELS SO FAR:\n")

comparison = pd.DataFrame({
    'Model': ['Logistic Regression', 'Decision Tree', 'Random Forest'],
    'Accuracy':  [accuracy,    dt_accuracy,  rf_accuracy],
    'Precision': [precision,   dt_precision, rf_precision],
    'Recall':    [recall,      dt_recall,    rf_recall],
    'F1-Score':  [f1,          dt_f1,        rf_f1]
})

# Format as percentages
for col in ['Accuracy', 'Precision', 'Recall', 'F1-Score']:
    comparison[col] = comparison[col].apply(lambda x: f"{x:.2%}")

print(comparison.to_string(index=False))

print(f"\n🎯 Priority Metric: RECALL (catching churners)")
print(f"Target: Recall > 75%")

📊 MODEL COMPARISON — ALL MODELS SO FAR:

              Model Accuracy Precision Recall F1-Score
Logistic Regression   80.31%    64.74% 56.95%   60.60%
      Decision Tree   77.83%    58.07% 59.63%   58.84%
      Random Forest   78.89%    62.88% 50.27%   55.87%

🎯 Priority Metric: RECALL (catching churners)
Target: Recall > 75%


## Day 12 Summary — Decision Tree & Random Forest

### Model Results
| Model | Accuracy | Precision | Recall | F1 |
|-------|----------|-----------|--------|----|
| Logistic Regression | 80.31% | 64.74% | 56.95% | 60.60% |
| Decision Tree | 77.83% | 58.07% | 59.63% | 58.84% |
| Random Forest | 78.89% | 62.88% | 50.27% | 55.87% |

### Key Findings
- Best Recall: Decision Tree at 59.63%
- Best F1: Logistic Regression at 60.60%
- Top churn predictor (feature importance): Decision Tree - 59.63%

### Next Steps
- Day 13: XGBoost + final model comparison
- Decide best model based on Recall > 75% target

In [3]:
# Calculate scale_pos_weight for class imbalance
# Formula: number of negatives (Stay) / number of positives (Churn)
neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
scale = neg / pos

print(f"Stay (0): {neg}, Churn (1): {pos}")
print(f"scale_pos_weight: {scale:.2f}")

# Initialize XGBoost
xgb_model = XGBClassifier(
    n_estimators=100,       # Number of trees
    max_depth=5,            # Depth per tree
    learning_rate=0.1,      # How much each tree corrects mistakes
    scale_pos_weight=scale, # Fixes class imbalance
    random_state=42,
    eval_metric='logloss',
    verbosity=0
)

# Train
xgb_model.fit(X_train, y_train)

# Predict
xgb_pred = xgb_model.predict(X_test)

print("\n✅ XGBoost trained!")

Stay (0): 4130, Churn (1): 1495
scale_pos_weight: 2.76

✅ XGBoost trained!


In [4]:
print("📊 MODEL PERFORMANCE — XGBoost:\n")

xgb_accuracy  = accuracy_score(y_test, xgb_pred)
xgb_precision = precision_score(y_test, xgb_pred, zero_division=0)
xgb_recall    = recall_score(y_test, xgb_pred, zero_division=0)
xgb_f1        = f1_score(y_test, xgb_pred, zero_division=0)

print(f"Accuracy:  {xgb_accuracy:.2%}")
print(f"Precision: {xgb_precision:.2%}")
print(f"Recall:    {xgb_recall:.2%}")
print(f"F1-Score:  {xgb_f1:.2%}")

print(f"\n🔢 CONFUSION MATRIX:\n")
xgb_cm = confusion_matrix(y_test, xgb_pred, labels=[0, 1])
xgb_cm_df = pd.DataFrame(
    xgb_cm,
    index=['Actual: Stay (0)', 'Actual: Churn (1)'],
    columns=['Predicted: Stay (0)', 'Predicted: Churn (1)']
)
print(xgb_cm_df)

print(f"\n📋 FULL CLASSIFICATION REPORT:\n")
print(classification_report(y_test, xgb_pred,
                             labels=[0, 1],
                             target_names=['Stay (0)', 'Churn (1)'],
                             zero_division=0))

📊 MODEL PERFORMANCE — XGBoost:

Accuracy:  73.56%
Precision: 50.17%
Recall:    78.61%
F1-Score:  61.25%

🔢 CONFUSION MATRIX:

                   Predicted: Stay (0)  Predicted: Churn (1)
Actual: Stay (0)                   741                   292
Actual: Churn (1)                   80                   294

📋 FULL CLASSIFICATION REPORT:

              precision    recall  f1-score   support

    Stay (0)       0.90      0.72      0.80      1033
   Churn (1)       0.50      0.79      0.61       374

    accuracy                           0.74      1407
   macro avg       0.70      0.75      0.71      1407
weighted avg       0.80      0.74      0.75      1407



In [5]:
print("🔧 RE-TRAINING ALL MODELS WITH class_weight='balanced'...\n")

# Logistic Regression balanced
lr_bal = LogisticRegression(
    random_state=42,
    max_iter=1000,
    class_weight='balanced'   # ← The fix
)
lr_bal.fit(X_train, y_train)
lr_bal_pred = lr_bal.predict(X_test)

# Decision Tree balanced
dt_bal = DecisionTreeClassifier(
    random_state=42,
    max_depth=5,
    class_weight='balanced'   # ← The fix
)
dt_bal.fit(X_train, y_train)
dt_bal_pred = dt_bal.predict(X_test)

# Random Forest balanced
rf_bal = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    max_depth=10,
    class_weight='balanced'   # ← The fix
)
rf_bal.fit(X_train, y_train)
rf_bal_pred = rf_bal.predict(X_test)

print("✅ All models re-trained with class balancing!")

🔧 RE-TRAINING ALL MODELS WITH class_weight='balanced'...

✅ All models re-trained with class balancing!


In [6]:
print("📊 FINAL MODEL COMPARISON:\n")

models = {
    'LR (original)':   lr_bal_pred,
    'LR (balanced)':   lr_bal_pred,
    'DT (original)':   dt_bal_pred,
    'DT (balanced)':   dt_bal_pred,
    'RF (original)':   rf_bal_pred,
    'RF (balanced)':   rf_bal_pred,
    'XGBoost':         xgb_pred,
}

# Original predictions (from Day 12)
original_preds = {
    'LR (original)': [0.8031, 0.6474, 0.5695, 0.6060],
    'DT (original)': [0.7783, 0.5807, 0.5963, 0.5884],
    'RF (original)': [0.7889, 0.6288, 0.5027, 0.5587],
}

# Build comparison table
results = []

# Add original results
for name, scores in original_preds.items():
    results.append({
        'Model': name,
        'Accuracy':  f"{scores[0]:.2%}",
        'Precision': f"{scores[1]:.2%}",
        'Recall':    f"{scores[2]:.2%}",
        'F1':        f"{scores[3]:.2%}"
    })

# Add balanced + XGBoost results
new_models = [
    ('LR (balanced)', lr_bal_pred),
    ('DT (balanced)', dt_bal_pred),
    ('RF (balanced)', rf_bal_pred),
    ('XGBoost',       xgb_pred),
]

for name, pred in new_models:
    results.append({
        'Model':     name,
        'Accuracy':  f"{accuracy_score(y_test, pred):.2%}",
        'Precision': f"{precision_score(y_test, pred, zero_division=0):.2%}",
        'Recall':    f"{recall_score(y_test, pred, zero_division=0):.2%}",
        'F1':        f"{f1_score(y_test, pred, zero_division=0):.2%}"
    })

comparison_df = pd.DataFrame(results)
print(comparison_df.to_string(index=False))

print(f"\n🎯 Target: Recall > 75%")

📊 FINAL MODEL COMPARISON:

        Model Accuracy Precision Recall     F1
LR (original)   80.31%    64.74% 56.95% 60.60%
DT (original)   77.83%    58.07% 59.63% 58.84%
RF (original)   78.89%    62.88% 50.27% 55.87%
LR (balanced)   72.71%    49.17% 79.14% 60.66%
DT (balanced)   70.58%    46.81% 78.34% 58.60%
RF (balanced)   77.19%    55.21% 75.13% 63.65%
      XGBoost   73.56%    50.17% 78.61% 61.25%

🎯 Target: Recall > 75%


In [7]:
print("🏆 BEST MODEL SELECTION:\n")

# Find best recall among all models
all_models = [
    ('LR (original)',  0.5695),
    ('DT (original)',  0.5963),
    ('RF (original)',  0.5027),
    ('LR (balanced)',  recall_score(y_test, lr_bal_pred, zero_division=0)),
    ('DT (balanced)',  recall_score(y_test, dt_bal_pred, zero_division=0)),
    ('RF (balanced)',  recall_score(y_test, rf_bal_pred, zero_division=0)),
    ('XGBoost',        recall_score(y_test, xgb_pred,   zero_division=0)),
]

best_name, best_recall = max(all_models, key=lambda x: x[1])

print(f"Best Recall: {best_name} at {best_recall:.2%}")

if best_recall >= 0.75:
    print(f"✅ Target achieved! {best_name} hits Recall > 75%")
else:
    print(f"⚠️ Target not yet reached. Best so far: {best_recall:.2%}")
    print(f"Gap to target: {0.75 - best_recall:.2%}")
    print(f"Next step: Model tuning in Week 3 will close this gap")

🏆 BEST MODEL SELECTION:

Best Recall: LR (balanced) at 79.14%
✅ Target achieved! LR (balanced) hits Recall > 75%


## Day 13 Summary — XGBoost + Final Model Comparison

### All Model Results
| Model | Accuracy | Precision | Recall | F1 |
|-------|----------|-----------|--------|----|
| LR (original)  | 80.31% | 64.74% | 56.95% | 60.60% |
| DT (original)  | 77.83% | 58.07% | 59.63% | 58.84% |
| RF (original)  | 78.89% | 62.88% | 50.27% | 55.87% |
| LR (balanced)  | 72.71% | 49.17% | 79.14% | 60.66% |
| DT (balanced)  | 70.58% | 46.81% | 78.34% | 58.60% |
| RF (balanced)  | 77.19% | 55.21% |75.13%  | 63.65% |
|      XGBoost   | 73.56% | 50.17% | 78.61% | 61.25% |

### Best Model
- Best Recall: LR (Logistic Regression) at 79.14%
- Target (75%) achieved: Yes

### Key Finding
class_weight='balanced' improved Recall by an average of ~22% across all 3 models because the original models were biased toward predicting Stay. By penalizing the model harder for missing churners, it stopped defaulting to the safe prediction and actively learned to identify churn patterns. The trade-off was a drop in Accuracy and Precision — but for churn prediction, that's an acceptable cost.

### Next Steps
- Week 3: Hyperparameter tuning to push Recall past 75%